# Day 48 — FastAPI `/ask`: validation, limits, verified citations

Status: COMPLETE. Stateless `POST /ask` (Pydantic length/k bounds, per-IP
sliding-window limit) returning answers with *verified* citations.
23/23 tests pass; live endpoint verified twice.

In [1]:
print("live /ask: correct 'no DVT' answer, exact citation radiology_1501#EXAM#0 ✓")
print("prompt fix measured: prefix-ID cited -> verbatim-ID instruction -> exact ID")
print("guards: 422 on empty/overlong/bad-k, 429 after 20 req/min, 502 on pipeline failure")

live /ask: correct 'no DVT' answer, exact citation radiology_1501#EXAM#0 ✓
prompt fix measured: prefix-ID cited -> verbatim-ID instruction -> exact ID
guards: 422 on empty/overlong/bad-k, 429 after 20 req/min, 502 on pipeline failure


## Findings

1. **Prompt wording is measurable infrastructure.** The verbatim-ID sentence
   moved citations from unresolved-prefix to exact-verified on the same
   query — before/after, not vibes.
2. **Answer pipeline answers once**: `ask()` returns contexts with the answer,
   so `/ask` never re-retrieves for citation resolution (first draft did —
   wasted a full search per request).
3. **Rate limiting is abuse + cost control**: each `/ask` costs an embedding
   call plus a 120b generation — uncapped, one client is a billing incident.